# Notebook 18 — Strict paired neutral-target pilot assessment (N07 vs N17v2)
**Status: EXECUTED (NIQE + matching only; FER/biometrics pending).** No diffusion generation. This notebook builds an audited matched reference data set, calculates 24 N17v2 NIQE scores and paired NIQE differences, and reports full privacy/FER paired comparison **only after** matching N17 output predictions are supplied by identical frozen evaluators. It does not substitute a different biometric model or emotion classifier.

**Scope:** 12 held-out sources × seeds 42 and 123; N07 `flowedit_g5p5` vs N17v2 `neutral_expression_target_g5p5`; N12 V3 corrected cosine and N16 NIQE for reference. Confidence intervals source-clustered. Six emotions (2 source clusters each): no definitive per-emotion conclusions. All row-level data private.


In [ ]:
!pip -q install "pyiqa==0.1.13" "pandas>=2.2,<3" "Pillow>=10,<12"
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib,json, numpy as np,pandas as pd,torch
from PIL import Image
RUNS=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=RUNS/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
N12=RUNS/'nb12_v3_biometric_11faafd9beb1_20261013T211442695856Z_bcf37701'
N16=RUNS/'nb16_no_reference_niqe_20261013T225209Z'
N17=RUNS/'nb17v2_neutral_expression_12sources_2seeds_sd3pinned'
OUT=RUNS/'nb18_paired_neutral_pilot_analysis_v1';OUT.mkdir(exist_ok=True)
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=pd.read_csv(N07/'raw_results.csv'); privacy=pd.read_csv(N12/'raw_corrected_privacy.csv')
niqe_old=pd.read_csv(N16/'niqe_per_image_PRIVATE.csv')
rec=pd.read_csv(N17/'private_execution_receipts.csv'); assert len(rec)==24
assert len(raw)==600 and len(privacy)==600 and len(niqe_old)==600
assert set(rec.status)=={'VERIFIED'}
keys=['source_id','seed'];assert not rec.duplicated(keys).any()
assert rec.source_id.nunique()==12 and set(rec.seed)=={42,123}
assert (rec.groupby('source_id').size()==2).all()
assert set(rec.runner_commit)=={'85594626161caca9dafa68592c1ffa21db87a806'}
assert set(rec.sd3_revision)=={'ea42f8cef0f178587cf766dc8129abd379c90671'}
for row in rec.itertuples(index=False):
 p=Path(row.output_path)
 assert p.is_file() and sha(p)==row.output_sha256, f'Invalid frozen generated image: {p}'
 with Image.open(p) as im: im.verify()
print('PREFLIGHT PASS: 24 persisted outputs, 12 sources, 2 seeds, hashes/model IDs verified')


In [ ]:
baseline=raw.query("condition == 'flowedit_g5p5'").copy()
cols=keys+['emotion_preserved','original_predicted_class','output_predicted_class','human_target_emotion','emotion_status','prepared_source_sha256']
assert not baseline.duplicated(keys).any()
baseline=baseline[cols]
ref=privacy.query("condition == 'flowedit_g5p5'")[keys+['arcface_cosine','auraface_cosine','pair_valid_arcface','pair_valid_auraface']]
assert not ref.duplicated(keys).any()
q=niqe_old.query("condition == 'flowedit_g5p5'")[keys+['niqe','original_niqe']]
assert not q.duplicated(keys).any()
paired=rec[keys+['emotion','output_sha256','source_sha256','output_path']].merge(baseline,on=keys,validate='one_to_one').merge(ref,on=keys,validate='one_to_one').merge(q,on=keys,validate='one_to_one')
assert len(paired)==24
assert (paired.source_sha256==paired.prepared_source_sha256).all()
assert paired.pair_valid_arcface.all() and paired.pair_valid_auraface.all()
assert set(paired.emotion_status)=={'success'}
assert paired.emotion.eq(paired.human_target_emotion).all()
paired.rename(columns={'arcface_cosine':'n07_arcface_cosine','auraface_cosine':'n07_auraface_cosine','niqe':'n07_niqe','emotion_preserved':'n07_emotion_preserved','output_predicted_class':'n07_predicted_class','original_predicted_class':'original_fer_predicted_class'},inplace=True)
paired.to_csv(OUT/'private_paired_reference_and_n17_manifest.csv',index=False)
print('MATCHING PASS:',len(paired),'paired rows. Baseline privacy corrected N12; emotion frozen N07; NIQE N16.')
print('N07 pilot-only reference metrics:')
print(paired[['n07_arcface_cosine','n07_auraface_cosine','n07_niqe','n07_emotion_preserved']].mean(numeric_only=True).to_string())


In [ ]:
import pyiqa
metric=pyiqa.create_metric('niqe',device='cpu');metric.eval()
def niqe(p):
 with Image.open(p) as im: rgb=np.asarray(im.convert('RGB'),dtype=np.float32)/255.
 ten=torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0)
 with torch.no_grad(): score=float(metric(ten).detach().cpu().item())
 assert np.isfinite(score)
 return score
cache=OUT/'private_n17_niqe.csv'
known=pd.read_csv(cache) if cache.exists() else pd.DataFrame(columns=keys+['output_sha256','n17_niqe'])
rows=known.to_dict('records')
for i,r in enumerate(paired.itertuples(index=False),1):
 matches=known[(known.source_id==r.source_id)&(known.seed==r.seed)&(known.output_sha256==r.output_sha256)] if len(known) else pd.DataFrame()
 if len(matches)==1:continue
 value=niqe(r.output_path)
 rows=[x for x in rows if not(x['source_id']==r.source_id and int(x['seed'])==r.seed)]
 rows.append({'source_id':r.source_id,'seed':int(r.seed),'output_sha256':r.output_sha256,'n17_niqe':value})
 pd.DataFrame(rows).to_csv(cache,index=False)
 print(f'NIQE {i}/24 processed',flush=True)
new=pd.read_csv(cache)
assert len(new)==24 and not new.duplicated(keys).any()
paired=paired.merge(new,on=keys+['output_sha256'],validate='one_to_one')
paired['delta_niqe']=paired.n17_niqe-paired.n07_niqe
print('NIQE paired delta N17 - N07:',paired.delta_niqe.mean(),'(lower typically means better natural-image statistics, not greater facial realism)')


In [ ]:
# The following is an explicit evaluator contract. Do not replace N07 FER or N12 biometric evaluators with packages that happen to be installed.
# Save predictions into this private CSV after running the exact frozen evaluators on N17 images.
NEEDED=OUT/'n17_exact_evaluator_outputs_PRIVATE.csv'
if not NEEDED.exists():
 template=paired[keys+['output_sha256']].copy()
 for c in ['n17_emotion_predicted_class','n17_arcface_cosine','n17_auraface_cosine','arcface_valid','auraface_valid','fer_evaluator_provenance','biometric_evaluator_provenance']:
  template[c]=pd.NA
 template.to_csv(NEEDED,index=False)
 print('CREATED EXACT-EVALUATOR INPUT TEMPLATE:',NEEDED)
 print('Privacy: repeat N12 V3 face-detection/alignment/ArcFace/AuraFace pipeline unchanged on 24 images.')
 print('Emotion: repeat the exact N07 FER classifier preprocessing/checkpoint/label mapping on 24 images.')
 print('Stop: do not use N07 historical (incorrect) biometric columns or infer labels from prompts.')
if NEEDED.is_file():
 estimates=pd.read_csv(NEEDED)
 required=['n17_emotion_predicted_class','n17_arcface_cosine','n17_auraface_cosine','arcface_valid','auraface_valid','fer_evaluator_provenance','biometric_evaluator_provenance']
 ready=(len(estimates)==24 and estimates[required].notna().all().all() and not estimates.duplicated(keys).any())
else:ready=False
print('EXACT EVALUATOR COMPARISON READY:',ready)


In [ ]:
# Cluster bootstrap: paired at image/seed level, resampled by distinct original source.
rng=np.random.default_rng(18018)
def cluster_ci(df,col,B=10000):
 groups=[g[col].astype(float).to_numpy() for _,g in df.groupby('source_id',sort=True)]
 draws=np.empty(B)
 for i in range(B):
  which=rng.integers(0,len(groups),len(groups))
  draws[i]=np.concatenate([groups[k] for k in which]).mean()
 return [float(np.quantile(draws,.025)),float(np.quantile(draws,.975))]
summary=[{'metric':'NIQE N17 minus N07','n_pairs':24,'n_sources':12,'mean_delta':float(paired.delta_niqe.mean()),'ci95_low':cluster_ci(paired,'delta_niqe')[0],'ci95_high':cluster_ci(paired,'delta_niqe')[1]}]
if ready:
 assert estimates[keys].merge(paired[keys],on=keys,validate='one_to_one').shape[0]==24
 merged=paired.merge(estimates,on=keys+['output_sha256'],validate='one_to_one')
 assert set(merged.arcface_valid.astype(str).str.lower())=={'true'} and set(merged.auraface_valid.astype(str).str.lower())=={'true'}
 assert merged.fer_evaluator_provenance.nunique()==1 and merged.biometric_evaluator_provenance.nunique()==1
 merged['n17_emotion_preserved']=merged.n17_emotion_predicted_class.astype(str).str.lower().eq(merged.original_fer_predicted_class.astype(str).str.lower())
 for m in ['arcface_cosine','auraface_cosine']:
  merged['delta_'+m]=merged['n17_'+m]-merged['n07_'+m]
 merged['delta_emotion_preserved']=merged.n17_emotion_preserved.astype(int)-merged.n07_emotion_preserved.astype(int)
 for m in ['arcface_cosine','auraface_cosine','emotion_preserved']:
  v=merged['delta_'+m];ci=cluster_ci(merged,'delta_'+m)
  summary.append({'metric':m+' N17 minus N07','n_pairs':24,'n_sources':12,'mean_delta':float(v.mean()),'ci95_low':ci[0],'ci95_high':ci[1]})
 merged.to_csv(OUT/'private_full_paired_evaluation.csv',index=False)
 print('FULL EXACT-EVALUATOR PAIRED ANALYSIS COMPLETE')
else:
 print('PRIVACY/FER INFERENCE BLOCKED: exact evaluator N17 scores not yet computed/validated.')
pd.DataFrame(summary).to_csv(OUT/'private_paired_summary.csv',index=False)
print(pd.DataFrame(summary).to_string(index=False))
status={'niqe':'EXECUTED','n17_exact_fer_and_v3_biometrics':'VALIDATED' if ready else 'PENDING','pilot_n_sources':12,'n_pairs':24,'caveat':'Source-cluster CIs, 12 sources only. No calibrated reidentification risk. No clinical inference.'}
(OUT/'evidence_status.json').write_text(json.dumps(status,indent=2))
print('PRIVATE ARTIFACTS:',OUT)
